# Data quality, missingness, and anomalies

**P0 Essential · D2 Independent · 90 minutes**

In [ ]:
import pandas as pd

from pathlib import Path

def locate(relative: str, local_name: str | None = None) -> Path:
    candidates = []
    if local_name:
        candidates.append(Path.cwd() / local_name)
    candidates.extend(root / relative for root in [Path.cwd(), *Path.cwd().parents])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Cannot find {relative}. Run from the course clone or place the downloaded data beside this notebook."
    )


In [ ]:
path = locate('datasets/teaching/air-quality/observations.csv', 'observations.csv')
clean = pd.read_csv(path).head(80)
quality = clean.copy()
quality['temperature_unit'] = 'C'
quality.loc[3, ['TEMP', 'temperature_unit']] = [quality.loc[3, 'TEMP'] * 9 / 5 + 32, 'F']
quality.loc[7, 'wd'] = 'INVALID'
quality.loc[11, 'WSPM'] = -2.0
quality = pd.concat([quality, quality.iloc[[5]]], ignore_index=True)
quality.tail(3)

## Task

Audit identity, missingness, units, category vocabulary, range validity, and anomalies. Repair only rules justified by the provided contract.

In [ ]:
def audit_quality(frame: pd.DataFrame) -> dict[str, int]:
    key = ['station', 'year', 'month', 'day', 'hour']
    valid_wind = {'N','NNE','NE','ENE','E','ESE','SE','SSE','S','SSW','SW','WSW','W','WNW','NW','NNW'}
    return {
        'duplicate_rows': int(frame.duplicated(key, keep=False).sum()),
        'missing_pm25': int(frame['PM2.5'].isna().sum()),
        'non_celsius_rows': int(frame['temperature_unit'].ne('C').sum()),
        'invalid_wind': int((frame['wd'].notna() & ~frame['wd'].isin(valid_wind)).sum()),
        'negative_wind_speed': int(frame['WSPM'].lt(0).sum()),
    }

In [ ]:
audit = audit_quality(quality)
assert audit['duplicate_rows'] == 2
assert audit['non_celsius_rows'] == 1
assert audit['invalid_wind'] == 1
assert audit['negative_wind_speed'] == 1
audit

## Decisions

Convert the known Fahrenheit row, quarantine the invalid category and impossible negative speed, and distinguish the exact duplicate from a repeated event. Do not impute PM2.5 here: any learned imputation belongs inside the modelling pipeline.

In [ ]:
repaired = quality.copy()
is_f = repaired['temperature_unit'].eq('F')
repaired.loc[is_f, 'TEMP'] = (repaired.loc[is_f, 'TEMP'] - 32) * 5 / 9
repaired.loc[is_f, 'temperature_unit'] = 'C'
repaired = repaired.drop_duplicates(['station','year','month','day','hour'])
repaired.loc[~repaired['wd'].isin(clean['wd'].dropna().unique()), 'wd'] = pd.NA
repaired.loc[repaired['WSPM'].lt(0), 'WSPM'] = pd.NA
assert audit_quality(repaired)['duplicate_rows'] == 0

## Transfer

Write a data-contract table with rule, severity, observed count, decision, and evidence. Explain which finding blocks modelling and which requires sensitivity/domain review.